In [ ]:
!nvidia-smi

Sat Oct 17 16:52:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets

In [3]:
!pip install -q -U bitsandbytes transformers peft trl accelerate datasets

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
!pip install -q -U bitsandbytes
import bitsandbytes
print("bitsandbytes version:", bitsandbytes.__version__)

GPU available: True
bitsandbytes version: 0.50.2


In [4]:
import torch
from google.colab import drive
drive.mount('/content/drive')
from transformers import BitsAndBytesConfig, AutoTokenizer
from peft import AutoPeftModelForCausalLM

ADAPTER = "/content/drive/MyDrive/os_tutor_adapter_v2"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
peft_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER, quantization_config=bnb_config, device_map={"": 0})
peft_model.eval()
peft_model.config.use_cache = True

TEACHER_PROMPT = ("You are an Operating Systems teacher. Answer the question directly first, "
                  "then give a short real-life analogy or example, then end with one line "
                  "starting 'Quick check:' that asks the student a question. "
                  "Keep it under 120 words.")
print("ready")

Mounted at /content/drive


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

ready


In [ ]:
from google.colab import files
uploaded = files.upload()

for name in uploaded:
    print("=====", name)
    print(uploaded[name][:700].decode("utf-8"))

Saving synchronization_test.jsonl to synchronization_test.jsonl
Saving synchronization_train.jsonl to synchronization_train.jsonl
Saving synchronization_validation.jsonl to synchronization_validation.jsonl
===== synchronization_test.jsonl
{"id": "sync_seed_132", "topic": "condition_variables", "instruction": "A thread needs to sleep until a shared condition becomes true, and another thread changes the shared state. Why is a condition variable appropriate?", "input": "", "output": "A condition variable allows the first thread to sleep while waiting for the required condition. The other thread can change the shared state and signal the condition, allowing the waiting thread to wake and re-check the condition."}
{"id": "sync_seed_129", "topic": "locks", "instruction": "A programmer implements a lock using a shared flag with separate read and write operations. Two threads can both observe the flag as 0. What is the synchronizati
===== synchronization_train.jsonl
{"id": "sync_seed_076", "to

In [ ]:
import json
from datasets import Dataset

def load(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def to_messages(row):
    question = row["instruction"]
    if row["input"].strip():
        question += "\n\n" + row["input"]
    return {"messages": [
        {"role": "user", "content": question},
        {"role": "assistant", "content": row["output"]},
    ]}

train_rows = load("synchronization_train_teacher.jsonl")
val_rows = load("synchronization_validation_teacher.jsonl")
test_rows = load("synchronization_test.jsonl")
print(len(train_rows), len(val_rows), len(test_rows))

train_ds = Dataset.from_list([to_messages(r) for r in train_rows])
val_ds = Dataset.from_list([to_messages(r) for r in val_rows])
print(train_ds[0])

110 15 15
{'messages': [{'role': 'user', 'content': 'What are the four conditions that must hold for deadlock to occur?'}, {'role': 'assistant', 'content': 'Deadlock needs four conditions to hold at the same time: mutual exclusion (a resource can be used by only one thread at a time), hold-and-wait (a thread holds one resource while waiting for another), no preemption (resources cannot be forcibly taken away), and circular wait (a ring of threads, each waiting for a resource held by the next). Think of four cars at a crossroads, each blocking the next one. The useful consequence is that removing any one condition makes deadlock impossible. Quick check: if every thread must acquire locks in the same global order, which of the four conditions are you removing?'}]}


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print(round(torch.cuda.memory_allocated() / 1e9, 1), "GB of GPU memory used")

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

2.7 GB of GPU memory used


In [ ]:
text = tokenizer.apply_chat_template(train_ds[0]["messages"], tokenize=False)
print(text)

<|im_start|>user
What are the four conditions that must hold for deadlock to occur?<|im_end|>
<|im_start|>assistant
<think>

</think>

Deadlock needs four conditions to hold at the same time: mutual exclusion (a resource can be used by only one thread at a time), hold-and-wait (a thread holds one resource while waiting for another), no preemption (resources cannot be forcibly taken away), and circular wait (a ring of threads, each waiting for a resource held by the next). Think of four cars at a crossroads, each blocking the next one. The useful consequence is that removing any one condition makes deadlock impossible. Quick check: if every thread must acquire locks in the same global order, which of the four conditions are you removing?<|im_end|>



In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir="os_tutor_adapter",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    max_length=512,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

Tokenizing train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [ ]:
for name, param in trainer.model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()

dtypes = {str(p.dtype) for p in trainer.model.parameters() if p.requires_grad}
print("trainable dtypes:", dtypes)

trainable dtypes: {'torch.float32'}


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,2.141594,1.648003,1.806665,0.603812,13113.000000
2,1.467694,1.416582,1.446997,0.628861,26226.000000
3,1.188451,1.359241,1.290476,0.647942,39339.000000


TrainOutput(global_step=42, training_loss=1.7517231929869879, metrics={'train_runtime': 147.0603, 'train_samples_per_second': 2.244, 'train_steps_per_second': 0.286, 'total_flos': 979978700974080.0, 'train_loss': 1.7517231929869879, 'epoch': 3.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

trainer.model.save_pretrained("/content/drive/MyDrive/os_tutor_adapter_v2")
tokenizer.save_pretrained("/content/drive/MyDrive/os_tutor_adapter_v2")
print("saved")

Mounted at /content/drive
saved


In [ ]:
import os, time
base = "/content/drive/MyDrive"
for d in sorted(os.listdir(base)):
    if d.startswith("os_tutor_adapter"):
        t = time.strftime("%H:%M", time.localtime(os.path.getmtime(f"{base}/{d}")))
        print(d, "| last changed:", t, "| files:", os.listdir(f"{base}/{d}"))

os_tutor_adapter | last changed: 18:01 | files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
os_tutor_adapter_v2 | last changed: 17:09 | files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
import json
from google.colab import files

TEACHER_PROMPT = ("You are an Operating Systems teacher. Answer the question directly first, "
                  "then give a short real-life analogy or example, then end with one line "
                  "starting 'Quick check:' that asks the student a question. "
                  "Keep it under 120 words.")

peft_model = trainer.model
peft_model.eval()
peft_model.config.use_cache = True

def answer(question, use_adapter, system=None, max_new_tokens=300):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        if use_adapter:
            out = peft_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        else:
            with peft_model.disable_adapter():
                out = peft_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

results = []
for row in test_rows:
    q = row["instruction"]
    results.append({
        "id": row["id"],
        "topic": row["topic"],
        "question": q,
        "reference": row["output"],
        "base": answer(q, use_adapter=False),
        "base_teacher_prompt": answer(q, use_adapter=False, system=TEACHER_PROMPT),
        "finetuned_v2": answer(q, use_adapter=True),
    })
    print("done", len(results), "/", len(test_rows))

with open("comparison_v2.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
files.download("comparison_v2.json")

for name in ["base", "base_teacher_prompt", "finetuned_v2"]:
    qc = sum("quick check" in r[name].lower() for r in results)
    words = sum(len(r[name].split()) for r in results) / len(results)
    print(name, "| avg words:", round(words), "| has Quick check:", qc, "/", len(results))

done 1 / 15
done 2 / 15
done 3 / 15
done 4 / 15
done 5 / 15
done 6 / 15
done 7 / 15
done 8 / 15
done 9 / 15
done 10 / 15
done 11 / 15
done 12 / 15
done 13 / 15
done 14 / 15
done 15 / 15


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

base | avg words: 213 | has Quick check: 0 / 15
base_teacher_prompt | avg words: 66 | has Quick check: 15 / 15
finetuned_v2 | avg words: 68 | has Quick check: 15 / 15


In [ ]:
import os
print([f for f in os.listdir('.') if f.endswith('.jsonl')])

['synchronization_train_teacher.jsonl', 'synchronization_test.jsonl', 'synchronization_validation_teacher.jsonl']


In [5]:
import zipfile, glob, os, torch
from sentence_transformers import SentenceTransformer

zipfile.ZipFile("chunks.zip").extractall("chunks_data")
files = sorted(glob.glob("chunks_data/**/*__chunks.txt", recursive=True))
print("chunk files:", len(files))

chunks, sources = [], []
for f in files:
    text = open(f, encoding="utf-8").read()
    for part in text.split("--- CHUNK ")[1:]:
        body = part.split("\n", 1)[1].strip()
        if body:
            chunks.append(body)
            sources.append(os.path.basename(f).replace("__chunks.txt", ".txt"))
print("chunks:", len(chunks))

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
chunk_emb = embedder.encode(chunks, batch_size=64, normalize_embeddings=True,
                            convert_to_tensor=True, show_progress_bar=True)

def retrieve(question, k=3):
    q = embedder.encode(question, normalize_embeddings=True, convert_to_tensor=True)
    scores = chunk_emb @ q
    top = torch.topk(scores, k).indices.tolist()
    return [(chunks[i], sources[i], float(scores[i])) for i in top]

chunk files: 24
chunks: 2243


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

In [6]:
def rag_answer(question, use_adapter=True, system=None, max_new_tokens=300):
    hits = retrieve(question)
    material = "\n\n".join(f"[{i+1}] {c}" for i, (c, s, sc) in enumerate(hits))
    user = ("Use the course material below to answer the student's question. "
            "If the material does not cover it, say so.\n\n"
            f"Course material:\n{material}\n\nQuestion: {question}")
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user})
    prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        if use_adapter:
            out = peft_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        else:
            with peft_model.disable_adapter():
                out = peft_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return text, hits

ans, hits = rag_answer("What is a race condition?")
print(ans)
print()
for c, s, sc in hits:
    print(s, round(sc, 3))

A race condition is a problem that arises when two or more threads execute a critical section at the same time, and their actions depend on each other. This can lead to an unpredictable result, like a counter that sometimes increases by 1 and sometimes by 2. Quick check: what is the simplest thing you can do to avoid this?

chunks\threads-intro__pdf.txt 0.695
chunks\Operating Systems - Three Easy Pieces_removed__pdf.txt 0.672
chunks\Operating Systems - Three Easy Pieces_removed__pdf.txt 0.668


In [7]:
import json
from google.colab import files

test_rows = [json.loads(l) for l in open("synchronization_test.jsonl") if l.strip()]

results = []
for row in test_rows:
    q = row["instruction"]
    a_ft, hits = rag_answer(q, use_adapter=True)
    a_bt, _ = rag_answer(q, use_adapter=False, system=TEACHER_PROMPT)
    results.append({
        "id": row["id"],
        "question": q,
        "reference": row["output"],
        "rag_finetuned_v2": a_ft,
        "rag_base_teacher_prompt": a_bt,
        "sources": [{"file": s, "score": round(sc, 3)} for c, s, sc in hits],
    })
    print("done", len(results), "/", len(test_rows))

with open("comparison_v3.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
files.download("comparison_v3.json")

for name in ["rag_finetuned_v2", "rag_base_teacher_prompt"]:
    qc = sum("quick check" in r[name].lower() for r in results)
    words = sum(len(r[name].split()) for r in results) / len(results)
    print(name, "| avg words:", round(words), "| has Quick check:", qc, "/", len(results))

done 1 / 15
done 2 / 15
done 3 / 15
done 4 / 15
done 5 / 15
done 6 / 15
done 7 / 15
done 8 / 15
done 9 / 15
done 10 / 15
done 11 / 15
done 12 / 15
done 13 / 15
done 14 / 15
done 15 / 15


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

rag_finetuned_v2 | avg words: 76 | has Quick check: 15 / 15
rag_base_teacher_prompt | avg words: 74 | has Quick check: 15 / 15


In [8]:
import gradio as gr

def tutor(question):
    ans, hits = rag_answer(question, use_adapter=True)
    src = "\n\n".join(f"{i+1}. {s} (score {sc:.2f})\n{c[:300]}..."
                      for i, (c, s, sc) in enumerate(hits))
    return ans, src

demo = gr.Interface(
    fn=tutor,
    inputs=gr.Textbox(label="Ask an Operating Systems question", lines=2),
    outputs=[gr.Textbox(label="Tutor answer", lines=8),
             gr.Textbox(label="Sources used", lines=12)],
    title="OSTutorLLM",
    description="Retrieval + fine-tuned Qwen3-4B"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7d4876d36e11941f5e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
import gradio as gr

def tutor(question):
    hits = retrieve(question)
    src = "\n\n".join(f"{i+1}. {s} (score {sc:.2f})\n{c[:300]}..."
                      for i, (c, s, sc) in enumerate(hits))
    base_ans, _ = rag_answer(question, use_adapter=False, system=TEACHER_PROMPT)
    ft_ans, _ = rag_answer(question, use_adapter=True)
    return base_ans, ft_ans, src

demo = gr.Interface(
    fn=tutor,
    inputs=gr.Textbox(label="Ask an Operating Systems question", lines=2),
    outputs=[gr.Textbox(label="Base model + teacher prompt", lines=8),
             gr.Textbox(label="Fine-tuned model", lines=8),
             gr.Textbox(label="Sources used", lines=12)],
    title="OSTutorLLM",
    description="Retrieval + Qwen3-4B: prompted base vs fine-tuned, same sources"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ce87a2e447909d59e4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
RICH_PROMPT = ("You are a friendly Operating Systems teacher talking to a student who is confused. "
               "Start by restating the doubt in one simple sentence. Then explain step by step in plain words, "
               "using the course material, and always mention how locks or mutexes are involved when relevant. "
               "Give one concrete example and one real-life analogy. "
               "Finish with one 'Quick check:' question. Use 150 to 200 words.")

ans, hits = rag_answer("What do you mean by condition variable? Explain with a simple example.",
                       use_adapter=False, system=RICH_PROMPT, max_new_tokens=400)
print(ans)

A condition variable (CV) is a synchronization tool that allows threads to wait for a specific condition to be true before proceeding. It’s used when a thread needs to wait until another thread signals that a condition is met.

For example, consider a parent thread waiting for a child thread to finish a task. The parent uses a CV to wait until the child signals that it’s done. The child, after finishing, signals the CV, waking up the parent.

Real-life analogy: Imagine a restaurant where the chef waits for the waiter to finish clearing tables before starting to cook. The waiter uses a CV to signal when the tables are clear, allowing the chef to proceed.

**Quick check:** What happens if a thread calls `wait()` on a condition variable when the condition is already true?
